# Visualisation of EEG & Sleep scoring

## Import recordings

Load packages

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import scipy
from scipy import signal
import os
from ephyviewer import mkQApp, MainViewer, TraceViewer
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, InMemoryAnalogSignalSource, EpochEncoder,TimeFreqViewer,AnalogSignalSourceWithScatter, EpochEncoder_ABC
from matplotlib import cm
from matplotlib.colors import to_hex
import re
import mne
import pandas as pd
import numpy as np
import csv
from collections import defaultdict
import ast
import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import string
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
%matplotlib widget

Choose .edf file

In [ ]:
dpath = "//10.69.168.1/crnldata/forgetting/Aurelie/EpiKid/RawData"
fc1 = FileChooser(dpath,select_default=True, show_only_dirs = False, title = "<b>Choose an .edf file</b>", layout=widgets.Layout(width='100%'))
fc1.filter_pattern = '*.edf'
display(fc1)
def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return 
fc1.register_callback(update_my_folder)

Load STS file & scoring infos

In [ ]:
folder= Path(dpath)
file= folder.name[:-4]
sts_file = f"{folder.parent}/{file}.sts"

def extract_printable(binary_data):
    return ''.join(chr(b) for b in binary_data if 32 <= b <= 126 or b in (10, 13))  # keep letters, numbers, space, newline

with open(sts_file, "rb") as f:
    binary_data = f.read()
text_data = extract_printable(binary_data)
keep_words = ["BStage N1", "BStage N2", "BStage N3", "BREM", "BWAKE", "BUnstaged"]
clean_text = re.sub(r"[^\x20-\x7E\n]", "", text_data)
matches = []
lines = clean_text.splitlines()
i = 0
while i < len(clean_text):
    for w in keep_words:
        if clean_text.startswith(w, i):
            matches.append(w[1:])
            i += len(w)
            break
    else:
        i += 1  # move forward if no match
print(f'{len(matches)} vigilance states found')

if "Advanced Artifact Rejection (1=ON, 0=OFF)" in clean_text:
    montagetype = "EPI"
elif "CReformatedMontageSLEEP" in clean_text:
    montagetype = "Sleep"
else:
    montagetype = "custum"
print(f'Montage type detected: {montagetype}')

Start of vigilance state scoring

In [ ]:
# Path to your text file
file_path = f"{folder.parent.parent}/Lag_VigScoring.txt"
target_name = file 
lag = None
with open(file_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # skip empty lines
        if ";" in line:
            name, number = line.split(";", 1)
            name = name.strip()
            number = number.strip()
            if name == target_name:
                try:
                    lag = float(number)
                except ValueError:
                    print(f"Error: Number for {name} is invalid: {number}")
                break  # stop after finding the first match

if lag is not None:
    print(f"Scoring started at {lag} s")
else:
    print(f"{target_name} not found in the file")
    lag = 0


Load and clean corrupted edf file (no montage)

In [ ]:
dpath = f"{folder.parent}/{file}.edf"
folder_base = Path(dpath) 

raw = mne.io.read_raw_edf(
    dpath,
    preload=True,
    verbose="error", 
    encoding='latin1'
)

raw.set_meas_date(None)
raw.set_annotations(mne.Annotations([], [], []))
raw.rename_channels(
    lambda ch: ch[:16].replace(" ", "_")
)
raw.pick_types(
    eeg=True,
    eog=True,
    emg=True,
    ecg=True,
    misc=True
)

data = raw.get_data()
info = raw.info.copy()

raw_clean = mne.io.RawArray(data, info)
mne.export.export_raw(
    f'{folder.parent}/{file}_repaired.edf',
    raw_clean,
    fmt="edf",
    physical_range=(-1000, 1000),
    overwrite=True
)
# reload to check
data = mne.io.read_raw_edf(f'{folder.parent}/{file}_repaired.edf', preload=False)
print(data)

Load metadata of edf

In [ ]:
raw_data = data.get_data() #data = mne.io.read_raw_edf(dpath, encoding='latin1')

info = data.info
channels = data.ch_names
timestamps = data.times
duration = data.duration
samplerate = data.info.get('sfreq')  # in Hz

print(data.info.get('meas_date'))
data.info.get('subject_info')
lowpass=data.info.get('lowpass')
highpass=data.info.get('highpass')

combined = raw_data.T
print(f'{round(np.shape(raw_data)[1]/samplerate/60/60,3)}h recording if sampled at {samplerate}Hz ({duration} seconds)')
print(f'{np.shape(raw_data)[0]} channels found: {channels}')

Save scoring for EphyViewer in .csv file

In [ ]:
SleepScoring= pd.DataFrame(matches, columns=['label'])
SleepScoring['time'] = list(range(int(lag), len(SleepScoring) * 30, 30))
SleepScoring['duration'] = 30
AutoSleepScoring_filename = os.path.join(Path(f'{folder.parent}'), f'EphyViewer_Scoring_{file}.csv')
SleepScoring.to_csv(AutoSleepScoring_filename, index=False)

Save scoring for Snooz in .tsv file

In [ ]:
def sec_to_hms(sec):
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{int(h)}:{int(m):02d}:{int(s):02d}"

def fill_unstaged_to_length(df, recording_length, epoch=30):
    df = df.sort_values("time").reset_index(drop=True)
    filled = []
    first_start = df.loc[0, "time"]
    if first_start > 0:
        filled.append({
            "label": "Unstaged",
            "time": 0,
            "duration": first_start
        })
    for i, r in df.iterrows():
        filled.append(r.to_dict())
        end_t = r["time"] + r["duration"]
        if i < len(df) - 1:
            next_t = df.loc[i + 1, "time"]
        else:
            next_t = recording_length
        if next_t > end_t:
            gap = next_t - end_t
            t = end_t
            while gap > 0:
                d = min(epoch, gap)
                filled.append({
                    "label": "Unstaged",
                    "time": t,
                    "duration": d
                })
                t += d
                gap -= d
    return pd.DataFrame(filled)

df_filled = fill_unstaged_to_length(SleepScoring, duration, epoch=30)

stage_map = {
    "WAKE": 0,
    "Stage N1": 1,
    "Stage N2": 2,
    "Stage N3": 3,
    "REM": 5,
    "Unstaged": 9,
}

out_df = pd.DataFrame({
    "group": ["stage"] * len(df_filled),
    "name": df_filled["label"].map(stage_map),
    "start_sec": df_filled["time"].astype(int),
    "duration_sec": df_filled["duration"].astype(int),
    "channels": ["[]"] * len(df_filled),
})

# Safety check for unmapped labels
if out_df["name"].isna().any():
    bad = df_filled.loc[out_df["name"].isna(), "label"].unique()
    raise ValueError(f"Unmapped labels found: {bad}")

# Save to TSV
out_df.to_csv(f"{folder.parent}/{file}_repaired_scoring.tsv", sep="\t", index=False)

Create & save a reapaired .edf with montage (for Snooz & SpikeDet)

In [ ]:
montage = 'Fp2-F4,F4-C4,C4-P4,P4-O2,Fp2-F8,F8-T4,T4-TB2,TB2-A2,A2-T6,T6-O2,Fp1-F3,F3-C3,C3-P3,P3-O1,Fp1-F7,F7-T3,T3-TB1,TB1-A1,A1-T5,T5-O1,Fz-Cz,Cz-Pz,EMG1+,EMG2+'
montage = 'Fp2-F4,F4-C4,C4-P4,P4-O2,Fp2-F8,F8-T4,T4-TB2,T6-O2,Fp1-F3,F3-C3,C3-P3,P3-O1,Fp1-F7,F7-T3,T3-TB1,T5-O1,Fz-Cz,Cz-Pz,EMG1+,EMG2+'
channels_clean = [ch.replace('EEG ', '') for ch in channels]
channels_clean = [ch.replace('EEG_', '') for ch in channels_clean]

pairs = montage.split(',')
montage_eeg=[]
montage_name=[]
for pair in pairs:
    if pair.count('-')==0:
        ch1 = pair
        idx1 = channels_clean.index(ch1)
        eeg=combined[:,idx1]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)
    else:        
        ch1, ch2 = pair.split('-')
        idx1 = channels_clean.index(ch1)
        idx2 = channels_clean.index(ch2)
        eeg=combined[:,idx2]-combined[:,idx1]        
        montage_eeg = eeg[:,np.newaxis] if len(montage_eeg) == 0 else np.append(montage_eeg, eeg[:,np.newaxis], axis=1)
        montage_name.append(pair)
        
montage_eegT = montage_eeg.T

info = mne.create_info(
    ch_names=montage_name,
    sfreq=samplerate,
    ch_types = ['eeg'] * (len(montage_name) - 2) + ['emg', 'emg']
)
raw_clean = mne.io.RawArray(montage_eegT, info)


outpath = Path(f"{folder.parent}/{file}_repaired_montage.edf")
mne.export.export_raw(
    outpath,
    raw_clean,
    fmt="edf",
    physical_range=(-1000, 1000),
    overwrite=True
)

# Reload to check
data_check = mne.io.read_raw_edf(outpath, preload=False)

## Display signals & vigilance states

Do montage for Ephyviewer (zscore & filter)

In [ ]:
"""
pairs = montage.split(',')
montage_eeg=[]
montage_name=[]
for pair in pairs:
    if pair.count('-')==0:
        ch1 = pair
        idx1 = channels_clean.index(ch1)
        eeg=combined[:,idx1]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)
    else:        
        ch1, ch2 = pair.split('-')
        idx1 = channels_clean.index(ch1)
        idx2 = channels_clean.index(ch2)
        eeg=combined[:,idx2]-combined[:,idx1]
        nyq = samplerate / 2
        f_lowcut = 0.5
        f_hicut = 50.0 #70
        Wn = [f_lowcut/nyq, f_hicut/nyq]
        sos = signal.butter(6, Wn, btype='band', output='sos')
        eeg = signal.sosfiltfilt(sos, eeg)
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        #montage_eeg = (eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, (eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)
eeg=[]
eeg_name=[]
for ch in channels_clean:
    idx1 = channels_clean.index(ch)
    eeg1=combined[:,idx1]
    eeg = zscore(eeg1[:,np.newaxis]) if len(eeg) == 0 else np.append(eeg, zscore(eeg1[:,np.newaxis]), axis=1)
    eeg_name.append(ch)

montage_name_eeg=montage_name[:-2]
"""

Check all channels (no montage)

In [ ]:
"""
app = mkQApp()
winlen = 10 # default window length in sec
t_start = 0.

win = MainViewer(debug=False, show_auto_scale=True)
view1 = TraceViewer.from_numpy(eeg, samplerate, t_start, 'Signals', channel_names=eeg_name)

cmap = cm.Spectral
values = np.linspace(0, 1, np.array(eeg).shape[1])
colormap = [to_hex(rgb) for rgb in cmap(values)]
for idx in range(np.array(eeg).shape[1]): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx]

view1.params['display_labels'] = True
view1.params['scale_mode'] = 'same_for_all'
view1.auto_scale()
view1.params['xsize'] = winlen
win.add_view(view1)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)

#Run
win.show()
app.exec()
"""

With sleep scoring (montage)

In [ ]:
"""
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{folder.parent}/EphyViewer_Scoring_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source =InMemoryAnalogSignalSource(montage_eeg, samplerate, t_start, channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'
view1.params['vline_color'] = "#FFFFFF00"

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#8b8b8b"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.params['vline_color'] = "#FFFFFF00"

view2.controls.hide()



# FFT
view3 = TimeFreqViewer(source=source, name='FFT')

view3.params['show_axis'] = True
view3.params['timefreq', 'f_start'] = 1
view3.params['timefreq', 'f_stop'] = 50
view3.params['timefreq', 'deltafreq'] = 1 #interval in Hz
view3.params['xsize'] = winlen
for idx, ch in enumerate(montage_name):
    #if ch == 'EMG': 
    #    view3.by_channel_params[f'ch{idx}', 'visible'] = False
    #else:        
    #    view3.by_channel_params[f'ch{idx}', 'clim'] = 1
    #    view3.by_channel_params[f'ch{idx}', 'visible'] = True
    if ch == 'C3-P3': 
        view3.by_channel_params[f'ch{idx}', 'clim'] = 1
        view3.by_channel_params[f'ch{idx}', 'visible'] = True
    else: 
        view3.by_channel_params[f'ch{idx}', 'visible'] = False



win.add_view(view1)
win.add_view(view2)
#win.add_view(view3)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'
"""

## Autosave scoring files (optional)

In [ ]:
"""
dpath = "//10.69.168.1/crnldata/forgetting/Aurelie/EpiKid/RawData/"
for completefile in os.listdir(dpath):
    if completefile.endswith(".edf"):
        folder= Path(dpath + completefile)
        file= completefile[:-4]
        print(file)
        sts_file = f"{folder.parent}/{file}.sts"

        def extract_printable(binary_data):
            return ''.join(chr(b) for b in binary_data if 32 <= b <= 126 or b in (10, 13))  # keep letters, numbers, space, newline

        with open(sts_file, "rb") as f:
            binary_data = f.read()
        text_data = extract_printable(binary_data)
        keep_words = ["BStage N1", "BStage N2", "BStage N3", "BREM", "BWAKE", "BUnstaged"]
        clean_text = re.sub(r"[^\x20-\x7E\n]", "", text_data)
        matches = []
        lines = clean_text.splitlines()
        i = 0
        while i < len(clean_text):
            for w in keep_words:
                if clean_text.startswith(w, i):
                    matches.append(w[1:])
                    i += len(w)
                    break
            else:
                i += 1  # move forward if no match
        print(f'{len(matches)} vigilance states found')


        # Path to your text file
        file_path = f"{folder.parent.parent}/Lag_VigScoring.txt"
        target_name = file 
        lag = None
        with open(file_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue  # skip empty lines
                if ";" in line:
                    name, number = line.split(";", 1)
                    name = name.strip()
                    number = number.strip()
                    if name == target_name:
                        try:
                            lag = float(number)
                        except ValueError:
                            print(f"Error: Number for {name} is invalid: {number}")
                        break  # stop after finding the first match

        if lag is not None:
            print(f"Scoring started at {lag} s")
        else:
            print(f"{target_name} not found in the file")
            lag = 0


        SleepScoring= pd.DataFrame(matches, columns=['label'])
        SleepScoring['time'] = list(range(int(lag), len(SleepScoring) * 30, 30))
        SleepScoring['duration'] = 30
        AutoSleepScoring_filename = os.path.join(Path(f'{folder.parent}'), f'EphyViewer_Scoring_{file}.csv')
        SleepScoring.to_csv(AutoSleepScoring_filename, index=False)



        def sec_to_hms(sec):
            h = sec // 3600
            m = (sec % 3600) // 60
            s = sec % 60
            return f"{int(h)}:{int(m):02d}:{int(s):02d}"

        def fill_unstaged_to_length(df, recording_length, epoch=30):
            df = df.sort_values("time").reset_index(drop=True)
            filled = []
            first_start = df.loc[0, "time"]
            if first_start > 0:
                filled.append({
                    "label": "Unstaged",
                    "time": 0,
                    "duration": first_start
                })
            for i, r in df.iterrows():
                filled.append(r.to_dict())
                end_t = r["time"] + r["duration"]
                if i < len(df) - 1:
                    next_t = df.loc[i + 1, "time"]
                else:
                    next_t = recording_length
                if next_t > end_t:
                    gap = next_t - end_t
                    t = end_t
                    while gap > 0:
                        d = min(epoch, gap)
                        filled.append({
                            "label": "Unstaged",
                            "time": t,
                            "duration": d
                        })
                        t += d
                        gap -= d
            return pd.DataFrame(filled)



        data = mne.io.read_raw_edf(
        f"{folder.parent}/{file}.edf",
        preload=True,
        verbose="error", 
        encoding='latin1'
        )
        raw_data = data.get_data() #data = mne.io.read_raw_edf(dpath, encoding='latin1')
        info = data.info
        channels = data.ch_names
        timestamps = data.times
        duration = data.duration
        samplerate = data.info.get('sfreq')  # in Hz
        print(f'{round(np.shape(raw_data)[1]/samplerate/60/60,3)}h recording if sampled at {samplerate}Hz ({duration} seconds)')




        df_filled = fill_unstaged_to_length(SleepScoring, duration, epoch=30)

        stage_map = {
            "WAKE": 0,
            "Stage N1": 1,
            "Stage N2": 2,
            "Stage N3": 3,
            "REM": 5,
            "Unstaged": 9,
        }

        out_df = pd.DataFrame({
            "group": ["stage"] * len(df_filled),
            "name": df_filled["label"].map(stage_map),
            "start_sec": df_filled["time"].astype(int),
            "duration_sec": df_filled["duration"].astype(int),
            "channels": ["[]"] * len(df_filled),
        })

        # Safety check for unmapped labels
        if out_df["name"].isna().any():
            bad = df_filled.loc[out_df["name"].isna(), "label"].unique()
            raise ValueError(f"Unmapped labels found: {bad}")

        # Save to TSV
        out_df.to_csv(f"{folder.parent}/{file}_repaired_scoring.tsv", sep="\t", index=False)
"""